# Minimal Token Sequence Test

Use this notebook to run tokens one by one, edit the sequence, and inspect the `State` after each step.

## Colab Setup

If you are running this from Colab, clone your repo first, then run the rest of the notebook from the repo root.

In [20]:
# Uncomment and edit these lines in Colab if needed.
!git clone https://github.com/chahineNejm/graph_Time_series
%cd YOUR_REPO

Cloning into 'graph_Time_series'...
remote: Enumerating objects: 179, done.
remote: Counting objects: 100% (179/179), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 179 (delta 81), reused 133 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (179/179), 173.45 KiB | 13.34 MiB/s, done.
Resolving deltas: 100% (81/81), done.
[Errno 2] No such file or directory: 'YOUR_REPO'
/content


## Load Current Token Files

This avoids importing the package-level `__init__`, so the notebook can run while the framework is still being assembled.

In [21]:
from pathlib import Path
import importlib.util
import sys
import types

import numpy as np


def find_repo_root():
    candidates = [Path.cwd()]
    candidates.extend(Path.cwd().glob("*"))
    for path in candidates:
        if (path / "graph_Time_series" / "state.py").exists():
            return path
    raise FileNotFoundError("Could not find repo root. Expected graph_Time_series/state.py under "
        "the current folder or one of its direct children."
    )


ROOT = find_repo_root()
PACKAGE_DIR = ROOT / "graph_Time_series"
print("Repo root:", ROOT)


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


pkg = types.ModuleType("graph_Time_series")
pkg.__path__ = [str(PACKAGE_DIR)]
sys.modules.setdefault("graph_Time_series", pkg)

blocks = types.ModuleType("graph_Time_series.token_blocks")
blocks.__path__ = [str(PACKAGE_DIR / "token_blocks")]
sys.modules.setdefault("graph_Time_series.token_blocks", blocks)

state_mod = load_module("graph_Time_series.state", PACKAGE_DIR / "state.py")
token_mod = load_module("graph_Time_series.token", PACKAGE_DIR / "token.py")
norm_mod = load_module(
    "graph_Time_series.token_blocks.normalization",
    PACKAGE_DIR / "token_blocks" / "normalization.py",
)
rbf_mod = load_module(
    "graph_Time_series.token_blocks.kernel_rbf",
    PACKAGE_DIR / "token_blocks" / "kernel_rbf.py",
)

State = state_mod.State
ZNormalizationToken = norm_mod.ZNormalizationToken
KernelRBFToken = rbf_mod.KernelRBFToken

print("Loaded token test modules.")

Repo root: /content/graph_Time_series
Loaded token test modules.


## Choose Tokens

In [22]:
TOKENS = {
    "ZNormalization": ZNormalizationToken(),
    "kernel_rbf": KernelRBFToken(),
}

# Edit this list to test your manual chain.
TOKEN_SEQUENCE = [
    "ZNormalization",
    "kernel_rbf",
]

## Load Electricity Data

This loads the GiftEval parquet electricity long config. A small holdout chunk is kept aside, and the token sequence runs on the working subset.

In [23]:
DATASET_NAME = "Salesforce/GiftEvalParquet"
DATA_CONFIG = "electricity_H_long"
MAX_RUN_SAMPLES = 24
HOLDOUT_SAMPLES = 8


def load_gifteval_electricity(max_run_samples=24, holdout_samples=8):
    import subprocess
    import sys

    try:
        from datasets import get_dataset_config_names, load_dataset
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
        from datasets import get_dataset_config_names, load_dataset

    configs = get_dataset_config_names(DATASET_NAME)
    if DATA_CONFIG not in configs:
        electricity_configs = [c for c in configs if "electricity" in c.lower()]
        print("Available electricity configs:", electricity_configs)
        raise ValueError(f"Config {DATA_CONFIG!r} was not found in {DATASET_NAME}.")

    print("Using dataset:", DATASET_NAME)
    print("Using config:", DATA_CONFIG)
    ds = load_dataset(DATASET_NAME, DATA_CONFIG, split="test")

    histories = []
    futures = []
    needed = max_run_samples + holdout_samples
    for row in ds:
        ts = np.asarray(row["target"], dtype=np.float32)
        if ts.size == 0:
            continue
        if not np.all(np.isfinite(ts)):
            fill = np.nanmean(ts) if np.any(np.isfinite(ts)) else 0.0
            ts = np.nan_to_num(ts, nan=fill, posinf=fill, neginf=fill)
        horizon = row.get("prediction_length")
        horizon = int(horizon) if horizon is not None else max(1, len(ts) // 5)
        if len(ts) < horizon + 10:
            continue
        histories.append(ts[:-horizon])
        futures.append(ts[-horizon:])
        if len(histories) >= needed:
            break

    min_hist = min(len(x) for x in histories)
    min_fut = min(len(x) for x in futures)
    H_all = np.asarray([x[-min_hist:] for x in histories], dtype=np.float32)
    F_all = np.asarray([x[:min_fut] for x in futures], dtype=np.float32)

    run_n = min(max_run_samples, len(H_all))
    H = H_all[:run_n]
    F = F_all[:run_n]
    H_holdout = H_all[run_n:run_n + holdout_samples]
    F_holdout = F_all[run_n:run_n + holdout_samples]
    return H, F, H_holdout, F_holdout


H, F, H_holdout, F_holdout = load_gifteval_electricity(
    MAX_RUN_SAMPLES, HOLDOUT_SAMPLES
)

print("Run data:", H.shape, F.shape)
print("Held-out data:", H_holdout.shape, F_holdout.shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Using dataset: Salesforce/GiftEvalParquet
Using config: electricity_H_long


electricity_H_long/train.parquet:   0%|          | 0.00/120M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1850 [00:00<?, ? examples/s]

ValueError: Unknown split "test". Should be one of ['train'].

## Run Sequence

In [ ]:
def feature_shapes(values):
    return {k: getattr(v, "shape", None) for k, v in values.items()}


def summarize_state(state, label):
    print(f"\n--- {label} ---")
    print(state)
    print("token_sequence:", state.token_sequence)
    print("class_counts:", state.class_counts)
    print("historical_features:", feature_shapes(state.historical_features))
    print("future_features:", feature_shapes(state.future_features))
    print("flags:", state.flags)
    print("transforms:", [t.name for t in state.transform_stack])
    print("prediction_names:", state.prediction_names)
    print("active_target_base:", state.active_target_base.shape)
    print("current_target:", state.current_target.shape)


def run_sequence(sequence, H, F):
    state = State(H, F)
    summarize_state(state, "after init")

    for name in sequence:
        token = TOKENS[name]
        print(f"\nToken: {name}")
        can_apply = token.can_apply(state)
        print("can_apply:", can_apply)
        if not can_apply:
            raise RuntimeError(f"Token {name} cannot apply to current state")
        state = token.apply(state)
        summarize_state(state, f"after {name}")

    forecast = state.get_final_prediction()
    print("\nFinal forecast shape:", forecast.shape)
    print("Final forecast sample:", np.round(forecast[0], 3))
    return state, forecast


state, forecast = run_sequence(TOKEN_SEQUENCE, H, F)

## Inspect State

In [ ]:
print("token_sequence:", state.token_sequence)
print("class_counts:", state.class_counts)
print("historical_features:", list(state.historical_features.keys()))
print("future_features:", list(state.future_features.keys()))
print("flags:", state.flags)
print("transforms:", [t.name for t in state.transform_stack])
print("prediction_names:", state.prediction_names)

In [ ]:
state.print_log()